# การบ้านสัปดาห์ 5 — "Prompt, ControlNet หรือ LoRA? ตัดสินด้วยตาและตัวเลข"
เลือก **วัตถุของคุณเอง** (สินค้า ของใช้ สัตว์เลี้ยง …) ถ่ายภาพ 5–8 ใบ ต่างมุม ต่างฉาก แล้วทำตามขั้นแล็บ 2–5 ของสัปดาห์ 5 (Week5_Lab_SD.ipynb)

**ส่ง (ก่อนคาบถัดไป)** notebook นี้ที่รันแล้ว · `my_refs/` ภาพอ้างอิง · `hw5_results.json` · memo 1 หน้า (ตอบว่าแนวทางไหนเหมาะกับงานคุณ เพราะอะไร — แปะภาพประกอบคู่ตัวเลข และบอกว่าตัวเลขไหนตอบคำถามไหน: CLIP ภาพ-ข้อความ = ตรง prompt · diversity = หลากหลาย · "ใบนี้ไหม" ตัดสินด้วยตา เพราะ CLIP ภาพ-ภาพ หยาบ ±3) · นำเสนอ 5 นาที

**เกณฑ์ 100 คะแนน** ภาพอ้างอิงและโจทย์ 15 · Prompt (ladder + ตัวเลข) 20 · ControlNet 15 · LoRA (ก่อน/หลัง + drift) 20 · ตารางเทียบ 4 แนวทาง 15 · memo + นำเสนอ 15 · โบนัส +5 (ทดลองเพิ่ม เช่น rank / step / prior preservation / วัด "ใบนี้ไหม" ด้วย DINO embedding)

> กติกาเดิม: seed เดียวกันทุกภาพ · เปลี่ยนทีละอย่าง · ตัวเลขทุกตัวมาจากการรันจริง · ถ้าตัวเลขขัดกับตา ให้เขียนไว้ตรง ๆ (เหมือนในแล็บ)

In [ ]:
# @snip:setup
!pip uninstall -q -y torchao
!pip install -q -U "diffusers>=0.30" "transformers>=4.45,<5" accelerate peft safetensors
import warnings, logging, os, math, time, json, random
warnings.filterwarnings("ignore"); logging.disable(logging.WARNING)
import torch, numpy as np
from PIL import Image, ImageDraw, ImageFilter
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่มี GPU!")
DEV = "cuda"; DT = torch.float16            # T4 ไม่รองรับ bf16 → ใช้ fp16 ทั้งวัน
SEED = 42                                    # seed เดียวกันทั้งวัน
def gen(seed=SEED): return torch.Generator(DEV).manual_seed(seed)

In [ ]:
# @snip:font
# ฟอนต์ไทยสำหรับกราฟ (Sarabun จาก GitHub google/fonts) — ถ้าโหลดไม่ได้จะใช้ฟอนต์ในเครื่องแทน ไม่ล้ม
import matplotlib, matplotlib.pyplot as plt, matplotlib.font_manager as fm
FONT_URL = "https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf"
try:
    if not os.path.exists("Sarabun-Regular.ttf"): os.system(f"wget -q {FONT_URL}")
    fm.fontManager.addfont("Sarabun-Regular.ttf"); plt.rcParams["font.family"] = "Sarabun"
except Exception as e:
    print("ใช้ฟอนต์สำรอง:", e)
plt.rcParams["axes.unicode_minus"] = False
os.makedirs("week5_fig", exist_ok=True)      # ภาพผลลัพธ์ทุกแล็บถูกบันทึกไว้ที่นี่ (ใช้ทำสไลด์)

## 1. ภาพอ้างอิงของคุณ
อัปโหลดภาพ 5–8 ใบ (jpg/png) → ถูกย่อเป็น 512×512 (ครอปกลางภาพ) · แก้ `CLASS` เป็นชื่อประเภทวัตถุภาษาอังกฤษ (เช่น mug, sneaker, cat)

In [ ]:
from google.colab import files
CLASS = "teapot"          # ← ประเภทของวัตถุคุณ (อังกฤษ คำเดียว)
TRIGGER = f"a sks {CLASS}"
up = files.upload(); os.makedirs("my_refs", exist_ok=True); REF_IMGS = []
for name, data in up.items():
    im = Image.open(__import__("io").BytesIO(data)).convert("RGB")
    w, h = im.size; s = min(w, h); im = im.crop(((w-s)//2, (h-s)//2, (w+s)//2, (h+s)//2)).resize((512, 512))
    im.save(f"my_refs/{len(REF_IMGS):02d}.jpg"); REF_IMGS.append(im)
print(len(REF_IMGS), "ภาพ")

In [ ]:
# @snip:helpers
def show_grid(imgs, titles=None, cols=None, size=2.6, save=None, suptitle=None):
    """แสดงภาพเป็นตาราง พร้อมป้ายไทย และบันทึกลง week5_fig/ ถ้าให้ชื่อ"""
    n = len(imgs); cols = cols or min(n, 6); rows = math.ceil(n / cols)
    fig, axs = plt.subplots(rows, cols, figsize=(size * cols, size * rows + (0.5 if suptitle else 0)))
    axs = np.array(axs).reshape(-1)
    for i, ax in enumerate(axs):
        ax.axis("off")
        if i < n:
            ax.imshow(imgs[i])
            if titles: ax.set_title(titles[i], fontsize=11)
    if suptitle: fig.suptitle(suptitle, fontsize=13)
    plt.tight_layout()
    if save: fig.savefig(f"week5_fig/{save}.png", dpi=120, bbox_inches="tight", facecolor="white")
    plt.show()

def free_mem(*names):
    """คืนหน่วยความจำ GPU ระหว่างแล็บ — T4 มี 15 GB โหลดทุกโมเดลพร้อมกันไม่ไหว"""
    import gc
    for n in names: globals().pop(n, None)
    gc.collect(); torch.cuda.empty_cache()
    print(f"GPU ใช้อยู่ {torch.cuda.memory_allocated()/1e9:.1f} GB")

def diversity(imgs, n_pairs=200, seed=0):
    """ความหลากหลาย = ระยะ L2 เฉลี่ยของคู่ภาพที่สุ่ม (ย่อเป็น 64×64 ก่อนวัด)"""
    x = np.stack([np.asarray(im.resize((64, 64)), dtype=np.float32) / 255 for im in imgs])
    rng = np.random.default_rng(seed); n = len(x); d = []
    for _ in range(n_pairs):
        i, j = rng.choice(n, 2, replace=False); d.append(np.sqrt(((x[i] - x[j]) ** 2).sum()))
    return float(np.mean(d))

show_grid(REF_IMGS, [f"อ้างอิง #{i}" for i in range(len(REF_IMGS))], save="hw_refs")

## 2. โหลด Stable Diffusion + CLIP (เหมือนแล็บ 1)

In [ ]:
# @snip:load_sd
from diffusers import StableDiffusionPipeline, DDIMScheduler
MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"      # mirror ของ runwayml (ถูกถอนไปแล้ว)
pipe = StableDiffusionPipeline.from_pretrained(MODEL, torch_dtype=DT, safety_checker=None).to(DEV)  # ปิดตัวกรอง NSFW (ประหยัด VRAM) — ใช้เฉพาะในห้องเรียน
pipe.set_progress_bar_config(disable=True)
for name in ["vae", "unet", "text_encoder", "scheduler"]:
    obj = getattr(pipe, name)
    n = sum(p.numel() for p in obj.parameters()) / 1e6 if hasattr(obj, "parameters") else 0
    print(f"{name:13s} {type(obj).__name__:28s} {n:7.0f} ล้านพารามิเตอร์")

In [ ]:
from diffusers import DPMSolverMultistepScheduler
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config); STEPS = 20
def to_tensor(im): return torch.from_numpy(np.asarray(im)).permute(2, 0, 1)[None].float() / 127.5 - 1

In [ ]:
# @snip:clip
from transformers import CLIPModel, CLIPProcessor
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEV).eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
@torch.no_grad()
def clip_feats(images=None, texts=None):
    out = {}
    if images is not None:
        f = clip.get_image_features(**clip_proc(images=images, return_tensors="pt").to(DEV)); out["img"] = f / f.norm(dim=-1, keepdim=True)
    if texts is not None:
        f = clip.get_text_features(**clip_proc(text=texts, return_tensors="pt", padding=True, truncation=True).to(DEV)); out["txt"] = f / f.norm(dim=-1, keepdim=True)
    return out
def clip_score(images, text):            # CLIP ภาพ-ข้อความ: ตรง prompt ไหม (0–100, ต่าง 1–2 = noise)
    f = clip_feats(images, [text]); return (100 * f["img"] @ f["txt"].T).squeeze(-1).cpu().numpy()
def clip_sim_to_refs(images, refs=REF_IMGS):   # CLIP ภาพ-ภาพ: เหมือนภาพอ้างอิงไหม (0–100, ต่าง ±3 = noise)
    f = clip_feats(images + refs)["img"]; return (100 * f[:len(images)] @ f[len(images):].T).mean(1).cpu().numpy()

## 3. Prompt ladder (แล็บ 2.1) — **แก้ LADDER ให้เป็นวัตถุของคุณ** วัดด้วย CLIP ภาพ-ข้อความ และอธิบายใน memo ว่าขั้นไหนช่วยมากสุด เพราะอะไร

In [ ]:
# @snip:ladder
LADDER = [
    ("P0 วัตถุ",       f"a {CLASS}"),
    ("P1 +ฉาก",        f"a {CLASS} on a wooden table in a cozy cafe"),
    ("P2 +สไตล์",      f"a {CLASS} on a wooden table in a cozy cafe, product photography"),
    ("P3 +แสง",        f"a {CLASS} on a wooden table in a cozy cafe, product photography, soft morning window light"),
    ("P4 +มุมกล้อง",   f"a {CLASS} on a wooden table in a cozy cafe, product photography, soft morning window light, close-up 50mm"),
    ("P5 +คุณภาพ",     f"a {CLASS} on a wooden table in a cozy cafe, product photography, soft morning window light, close-up 50mm, highly detailed, sharp focus"),
]
NEG = "blurry, text, watermark, deformed, extra parts"
def run(prompts, negative=NEG, guidance=7.5, seed=SEED, steps=None, **kw):
    return pipe(prompts, negative_prompt=[negative] * len(prompts), num_inference_steps=steps or STEPS,
                guidance_scale=guidance, generator=gen(seed), **kw).images
imgs = run([p for _, p in LADDER])
scores = [clip_score([im], LADDER[-1][1])[0] for im in imgs]
show_grid(imgs, [f"{n}\nCLIP {s:.1f}" for (n, _), s in zip(LADDER, scores)], save="lab2_ladder")

## 4. ControlNet Canny (แล็บ 3.3) — ใช้ภาพอ้างอิง #0 ของคุณเป็นโครงภาพ ลอง scale 0.3 / 0.6 / 1.0

In [ ]:
# @snip:controlnet
import cv2
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline
cn_canny = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny", torch_dtype=DT)
cnp = StableDiffusionControlNetPipeline.from_pipe(pipe, controlnet=cn_canny).to(DEV); cnp.set_progress_bar_config(disable=True)
def canny(im, lo=100, hi=200):
    e = cv2.Canny(np.asarray(im), lo, hi); return Image.fromarray(np.stack([e] * 3, -1))
edge = canny(REF_IMGS[0])
P_CN = f"a {CLASS} on a marble counter, luxury product photography"
imgs = [edge] + [cnp(P_CN, image=edge, controlnet_conditioning_scale=s, negative_prompt=NEG,
                     num_inference_steps=STEPS, generator=gen()).images[0] for s in [0.3, 0.6, 1.0]]
show_grid(imgs, ["โครง (Canny)"] + [f"scale {s}" for s in [0.3, 0.6, 1.0]], save="lab3_canny")

## 5. LoRA (แล็บ 4) — ก่อนเทรน / เทรน 400 step / หลังเทรน / ทดสอบ drift

In [ ]:
# @snip:before_lora
free_mem("cnp", "cn_canny", "cn_scr")
pipe.vae.to(DT); pipe.text_encoder.to(DT); pipe.unet.to(DT)   # คืนเป็น fp16 ทั้งชุด
TRIGGER = "a sks teapot"
P_TEST = [f"{TRIGGER} on a wooden table, product photography",
          f"{TRIGGER} on a beach at sunset", f"{TRIGGER}, watercolor illustration"]
before = run(P_TEST, seed=SEED)
print("CLIP ภาพ-ภาพ กับภาพอ้างอิง (ก่อนเทรน):", clip_sim_to_refs(before).round(1))
show_grid(before, ["ก่อนเทรน: โต๊ะไม้", "ก่อนเทรน: ชายหาด", "ก่อนเทรน: สีน้ำ"], save="lab4_before")

In [ ]:
# @snip:lora_setup
from peft import LoraConfig
unet = pipe.unet; unet.requires_grad_(False)
pipe.vae.requires_grad_(False); pipe.text_encoder.requires_grad_(False)
if getattr(unet, "peft_config", None):                     # รันซ้ำ: ถอด adapter เก่าออกก่อน และล้างสถานะเทรน
    unet.delete_adapters(list(unet.peft_config.keys())); unet.peft_config = {}; unet._hf_peft_config_loaded = False
globals().pop("lora_losses", None)
unet.add_adapter(LoraConfig(r=8, lora_alpha=8, init_lora_weights="gaussian",
                            target_modules=["to_q", "to_k", "to_v", "to_out.0"]))
lora_params = [p for p in unet.parameters() if p.requires_grad]
for p in lora_params: p.data = p.data.float()            # LoRA เก็บเป็น fp32 ฝั่งโมเดลเป็น fp16
n_tr = sum(p.numel() for p in lora_params); n_all = sum(p.numel() for p in unet.parameters())
print(f"เทรน {n_tr/1e6:.2f} ล้าน จาก {n_all/1e6:.0f} ล้าน = {100*n_tr/n_all:.2f}%")
unet.enable_gradient_checkpointing()

In [ ]:
# @snip:lora_train
STEPS_TRAIN, LR = 400, 1e-4
if "lora_losses" in globals() and len(lora_losses) >= STEPS_TRAIN:
    print("เทรนไปแล้ว — ข้าม (รัน 4.2 ใหม่ถ้าต้องการเทรนซ้ำ)")
else:
    lora_losses, snaps = [], []
    opt = torch.optim.AdamW(lora_params, lr=LR, weight_decay=1e-2)
    scaler = torch.cuda.amp.GradScaler()
    noise_sch = DDIMScheduler.from_config(pipe.scheduler.config)
    tok = pipe.tokenizer([TRIGGER], padding="max_length", max_length=77, return_tensors="pt")
    pipe.vae.to(DT); pipe.text_encoder.to(DT)                 # บาง pipeline ในแล็บ 3 เปลี่ยน VAE เป็น fp32 → บังคับกลับเป็น fp16
    with torch.no_grad():
        emb = pipe.text_encoder(tok.input_ids.to(DEV))[0]
        lat_refs = [pipe.vae.encode(to_tensor(im).to(DEV, pipe.vae.dtype)).latent_dist for im in REF_IMGS]
    unet.train(); t0 = time.time()
    for step in range(STEPS_TRAIN):
        z0 = (lat_refs[step % len(REF_IMGS)].sample() * pipe.vae.config.scaling_factor).to(DT)
        if random.random() < 0.5: z0 = z0.flip(-1)                       # augment: กลับซ้ายขวา
        noise = torch.randn_like(z0); t = torch.randint(0, 1000, (1,), device=DEV)
        zt = noise_sch.add_noise(z0, noise, t)
        with torch.autocast("cuda", dtype=DT):
            pred = unet(zt, t, encoder_hidden_states=emb).sample
            loss = torch.nn.functional.mse_loss(pred.float(), noise.float())
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        lora_losses.append(loss.item())
        if (step + 1) % 100 == 0:
            unet.eval()
            with torch.no_grad(), torch.autocast("cuda", dtype=DT):   # LoRA เป็น fp32 ระหว่างเทรน → autocast ตอนสร้างภาพ (ทดสอบบน T4 แล้ว)
                snaps.append(run([P_TEST[0]], seed=SEED)[0])
            unet.train()
            print(f"step {step+1:4d}  loss {np.mean(lora_losses[-100:]):.4f}  {time.time()-t0:.0f} วิ")
    unet.eval()
    for p in lora_params: p.data = p.data.to(DT)             # เทรนเสร็จ: เก็บ LoRA เป็น fp16 ให้ใช้กับ pipeline ได้ตรง ๆ ในแล็บถัดไป
    show_grid([before[0]] + snaps, ["ก่อนเทรน"] + [f"step {100*(i+1)}" for i in range(len(snaps))], save="lab4_snaps",
              suptitle=P_TEST[0])

In [ ]:
# @snip:after_lora
unet.to(DT)   # ให้ทุกพารามิเตอร์รวม LoRA เป็น fp16 ก่อนใช้กับ pipeline
after = run(P_TEST, seed=SEED)
print("ก่อน:", clip_sim_to_refs(before).round(1), " หลัง:", clip_sim_to_refs(after).round(1))
show_grid(before + after, ["ก่อน: โต๊ะไม้", "ก่อน: ชายหาด", "ก่อน: สีน้ำ", "หลัง: โต๊ะไม้", "หลัง: ชายหาด", "หลัง: สีน้ำ"], cols=3, save="lab4_before_after")
sweep = [run([P_TEST[1]], seed=SEED, cross_attention_kwargs={"scale": w})[0] for w in [0.0, 0.5, 1.0]]
show_grid(sweep, [f"น้ำหนัก LoRA {w}" for w in [0.0, 0.5, 1.0]], save="lab4_weight")
from peft.utils import get_peft_model_state_dict
StableDiffusionPipeline.save_lora_weights("my_lora", unet_lora_layers=get_peft_model_state_dict(unet))
print("ขนาด adapter:", os.path.getsize("my_lora/pytorch_lora_weights.safetensors") / 1e6, "MB")

In [ ]:
# @snip:drift
generic = run([f"a {CLASS} on a wooden table, product photography"] * 3, seed=SEED)
print("CLIP ภาพ-ภาพ กับภาพอ้างอิง ทั้งที่ไม่ได้สั่ง sks:", clip_sim_to_refs(generic).round(1))
show_grid(generic, ["'a teapot' #1", "'a teapot' #2", "'a teapot' #3"], save="lab4_drift", suptitle="language drift: ไม่ได้พิมพ์ sks ก็ยังได้วัตถุของคุณ")

## 6. ตารางเทียบ 4 แนวทาง (แล็บ 5) — ใช้โครงภาพ Canny จากข้อ 4 แทน scribble · ผลบันทึกลง `hw5_results.json`

In [ ]:
# @snip:final
cn_scr = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny", torch_dtype=DT)
cnp = StableDiffusionControlNetPipeline.from_pipe(pipe, controlnet=cn_scr).to(DEV); cnp.set_progress_bar_config(disable=True)
P_FINAL = "{} on a wooden table in a cozy cafe, empty wall on the right, product photography, soft morning light"
SEEDS = [42, 7, 123, 2024]
def gen_set(use_lora, use_cn):
    scale = 1.0 if use_lora else 0.0; prompt = P_FINAL.format(TRIGGER if use_lora else f"a {CLASS}"); out = []
    for s in SEEDS:
        if use_cn:
            out.append(cnp(prompt, image=edge, controlnet_conditioning_scale=1.0, negative_prompt=NEG,   # โครงภาพจาก 3.4 (14 px) scale เดียวกับ 3.4
                           num_inference_steps=STEPS, generator=gen(s), cross_attention_kwargs={"scale": scale}).images[0])
        else:
            out.append(pipe(prompt, negative_prompt=NEG, num_inference_steps=STEPS, generator=gen(s),
                            cross_attention_kwargs={"scale": scale}).images[0])
    return out
sets = {"A prompt": gen_set(False, False), "B +ControlNet": gen_set(False, True),
        "C +LoRA": gen_set(True, False), "D LoRA+ControlNet": gen_set(True, True)}
rows = []
for name, ims in sets.items():
    rows.append(dict(แนวทาง=name, CLIP_ภาพ_ข้อความ=clip_score(ims, P_FINAL.format(f"a {CLASS}")).mean(),
                     CLIP_ภาพ_ภาพ=clip_sim_to_refs(ims).mean(), diversity=diversity(ims, 50)))
import pandas as pd; df = pd.DataFrame(rows).round(1); print(df.to_string(index=False))
df.to_json("hw5_results.json", orient="records", force_ascii=False, indent=1)
show_grid([im for ims in sets.values() for im in ims], [f"{n} · seed {s}" for n in sets for s in SEEDS], cols=4, save="lab5_final")

## 7. (โบนัส) ทดลองเพิ่ม 1 อย่าง — เลือกข้อใดข้อหนึ่ง แล้วใส่ผลใน memo
- rank 4 vs 8 vs 16 · step 200 vs 400 vs 800 (ดู drift) · ใส่ prior preservation (เพิ่มภาพ `a {CLASS}` ทั่วไปที่โมเดลสร้างเองเข้าชุดเทรน) · ControlNet scale sweep ละเอียด · ใช้ DINO (`facebook/dinov2-small`) วัดภาพ-ภาพ แทน CLIP แล้วดูว่าแยก "ใบนี้" ได้ดีกว่าไหม